# MoE load-balancing lab

Why do Mixture-of-Experts routers pile tokens onto a few experts, and which fixes produce a model that actually uses
all of its experts? All CPU-scale results are already in `results/` (see `README.md`). This notebook is for the parts
that need a GPU or Hugging Face access, and for re-running anything.

| experiment | what | where it runs |
|---|---|---|
| E1 / E2 | mechanisms and fixes (synthetic) | CPU, results in repo |
| E4 / E5 / E6 | per-method tuning, pre-registered held-out test, depth & representation pathway | CPU, results in repo |
| E3 v2 `--preset cpu` | 4-layer byte-level MoE LM | CPU, results in repo |
| **E3 v2 `--preset gpu`** | 6-layer, 32-expert LM, 10k steps | **GPU (this notebook)**; all 21 runs in repo (Kaggle T4) |
| **E7** | routing over a real MoE's pretraining checkpoints (OLMoE) | **GPU + Hugging Face (this notebook)** |

## Environment check (Kaggle / Colab)

Run this first. On Kaggle it needs **Settings → Internet: On** (phone-verified account) and, for the GPU sections,
**Accelerator: GPU T4 ×2** (a P100 may not work with the preinstalled PyTorch). `torch`, `numpy`, `matplotlib`,
`pytest`, `transformers`, `accelerate` and `huggingface_hub` are preinstalled there, so the `pip install` below only
confirms them.

This cell checks the internet and the GPU, sends the Hugging Face cache to scratch space (`/kaggle/temp`), and sets
`E7_DTYPE` / `E7_MAX_CKPTS` for the E7 cell. T4/P100 GPUs have no native bf16, so E7 falls back to float16 there, and
the number of checkpoints is capped by free disk (each OLMoE revision is ~14 GB and the script never deletes them).
Kaggle sessions last at most 12 h and only `/kaggle/working` is kept as output; the clone cell below puts the repo
(and so `results/`) there. Use **Save Version → Save & Run All**: its output is kept even if it hits the time limit.
With T4 ×2 the scripts use both GPUs (one E3 run per GPU; E7 splits the model across both).

In [ ]:
import os, shutil, socket

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:
    os.makedirs("/kaggle/temp/hf", exist_ok=True)
    os.environ["HF_HOME"] = "/kaggle/temp/hf"  # big checkpoint downloads go to scratch, not the saved output

def reachable(host):
    try:
        socket.create_connection((host, 443), timeout=5).close()
        return True
    except OSError:
        return False

for host, why in [("github.com", "git clone"), ("raw.githubusercontent.com", "Shakespeare corpus (E3)"),
                  ("huggingface.co", "OLMoE checkpoints (E7)")]:
    print(f"{'ok     ' if reachable(host) else 'BLOCKED'} {host:28s} {why}")
if ON_KAGGLE:
    print("If any host is BLOCKED: Settings -> Internet -> On, then restart the session.")

import torch
E7_DTYPE, E7_MAX_CKPTS = "bfloat16", 10
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if N_GPUS:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB, sm_{props.major}{props.minor}, {N_GPUS} device(s)")
    try:
        (torch.zeros(1, device="cuda") + 1).item()
    except Exception as e:  # e.g. P100 (sm_60) with a PyTorch build that dropped it
        print("CUDA is visible but a kernel failed -> switch the accelerator to T4:", e)
    if props.major < 8:
        E7_DTYPE = "float16"
    if props.total_memory * N_GPUS < 24 * 2**30:
        print("WARNING: OLMoE-1B-7B is 13.8 GB in 16-bit; with < 24 GB of GPU memory in total E7 may run out of memory.")
else:
    print("No GPU: the GPU cells will skip themselves; only the CPU sections (quick look, re-runs, analysis) run.")
    print("On Kaggle: Settings -> Accelerator -> GPU T4 x2, then restart the session.")

hf_home = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.makedirs(hf_home, exist_ok=True)
free_gb = shutil.disk_usage(hf_home).free / 2**30
# one download per chosen revision plus `main`, ~15 GB each
E7_MAX_CKPTS = max(0, min(E7_MAX_CKPTS, int(free_gb // 15) - 1))
print(f"free disk for HF cache: {free_gb:.0f} GiB -> E7_MAX_CKPTS={E7_MAX_CKPTS}, E7_DTYPE={E7_DTYPE}")
if E7_MAX_CKPTS < 2:
    print("WARNING: not enough disk for a useful E7 run; skip E7 or free space / use a smaller MoE repo.")

In [ ]:
# Clone the repo. For a private repo, use a token: https://<TOKEN>@github.com/westhauxstripclub/colab.git
BRANCH = "claude/vigilant-curie-gn5azm"
!git clone -b {BRANCH} https://github.com/westhauxstripclub/colab.git moe-lab
%cd moe-lab
!pip -q install -r requirements.txt
!python -m pytest -q tests

In [ ]:
# Resume: copy finished runs from earlier versions' output. On Kaggle, "Add Input" -> Your Work -> this notebook's
# previous version; its /kaggle/working shows up under /kaggle/input/. Runs already in results/ are skipped by every
# script, so only unfinished ones run again.
import glob, os, shutil
copied = 0
srcs = [p for ext in ("json", "npz") for p in glob.glob(f"/kaggle/input/**/moe-lab/results/*/**/*.{ext}", recursive=True)]
for src in srcs:
    dst = os.path.join("results", src.split("/moe-lab/results/", 1)[1])
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
        copied += 1
print(f"copied {copied} finished run(s) from /kaggle/input")
for exp in ("e3v2_lm_gpu", "e7_real_checkpoints"):
    print(f"results/{exp}: {len(glob.glob(f'results/{exp}/**/*.json', recursive=True))} finished file(s)")

## Quick look: one run without balancing vs. one with a strong aux loss

In [ ]:
import sys; sys.path.insert(0, '.')
import matplotlib.pyplot as plt
from moe_lab.train_synthetic import SynthRunConfig, run_synthetic

base = dict(moe=dict(d_hidden=8, n_experts=32), task=dict(n_clusters=32, n_train=32000, n_val=8000), seed=0)
runs = {name: run_synthetic(SynthRunConfig(name=name, **{**base, 'moe': {**base['moe'], **extra}}))
        for name, extra in [('none', {}), ('aux_1e-1', dict(aux_loss_coef=0.1))]}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (name, r) in zip(axes, runs.items()):
    f = r['final']
    ax.bar(range(len(f['load'])), sorted(f['load'], reverse=True))
    ax.set_title(f"{name}: test loss {f['test_loss']:.3f}, dead experts {f['dead_experts']}")
    ax.set_xlabel('expert (sorted by load)'); ax.set_ylabel('share of tokens')
plt.show()

## When does routing settle in a real MoE? (E7, OLMoE checkpoints)

Needs Hugging Face access and ≥ 24 GB of GPU memory: one large GPU, or Kaggle T4 ×2 with `--device_map auto`
(OLMoE-1B-7B is 13.8 GB in 16-bit; each checkpoint takes ~2–3 min to download and load). It runs before the E3 grid.
Each finished checkpoint is saved at once under `results/e7_real_checkpoints/<repo>/`, so a crash or timeout loses
nothing and the next session resumes. The script frees GPU memory between checkpoints and prints per-GPU usage
(`loaded:` / `freed:` lines); with `--device_map` it leaves 3 GiB per card free for loading buffers. Tested in the repo only on tiny
randomly initialised OLMoE-architecture models. OLMoE was trained *with* load balancing, so this shows when routing
settles under balancing, not unbalanced collapse.

In [ ]:
if N_GPUS == 0:
    print("skipped: no GPU in this session")
else:
    E7_SPLIT = "--device_map auto" if N_GPUS >= 2 else "--device cuda"
    %cd experiments
    !pip -q install -r ../requirements-colab.txt
    !python e7_real_checkpoints.py --repo allenai/OLMoE-1B-7B-0924 --max_checkpoints {E7_MAX_CKPTS} --dtype {E7_DTYPE} {E7_SPLIT}
    %cd ..

## Scaled language model on a GPU (E3 v2, `gpu` preset)

6 layers, d_model 256, 32 experts × 128 (top-2), context 256, 10k steps, 3 seeds. Measured on one Kaggle T4:
~13.5 min per `dense_active` run, 36–42 min per MoE run, ~47 min per `dense_total` run, so the 7 methods below
(21 runs) take ~12.9 h on one GPU and ~6.5 h on T4 ×2 (one run per GPU). Finished runs are cached, so re-running
resumes; all 21 are already in `results/e3v2_lm_gpu/`, so this cell is a no-op unless you add methods or seeds.

**Caveat:** this preset makes ~55 passes over ~1.5 MB of training text, so val loss bottoms out around step 3000
and then rises in every run. Compare methods on the *best checkpoint* table in `results/summary_e3v2_gpu.md`, not on
final loss.

In [ ]:
if N_GPUS == 0:
    print("skipped: no GPU in this session")
else:
    %cd experiments
    !python e3_lm.py --preset gpu --device cuda --workers {N_GPUS} --threads 2 --methods none aux_1e-2 aux_1e-2_then_release frozen_router detach_input dense_active dense_total
    %cd ..

## Re-run the CPU experiments (optional; results are already in the repo)

Colab CPUs are slow; `--seeds 4` gives a quick pass. E5 must only be run after E4 `select` (see `PREREGISTRATION.md`).

In [ ]:
# !python e1_mechanisms.py --seeds 20 --workers 2
# !python e4_tuning.py tune && python e4_tuning.py select && python e4_tuning.py eval
# !python e5_heldout.py && python e6_deep.py
# !python e3_lm.py --preset cpu --workers 2

In [ ]:
%cd experiments
!python analyze.py && python analyze2.py
from IPython.display import Image, Markdown, display
import glob
for name in sorted(glob.glob('../results/summary_*.md')):
    display(Markdown(open(name).read()))
for p in sorted(glob.glob('../results/figures/*.png')):
    display(Image(p))
%cd ..

In [ ]:
# Bundle only the new outputs, to download from the Output tab and upload to the repo.
import os, zipfile
keep = ["results/e3v2_lm_gpu", "results/e7_real_checkpoints", "results/runs_e3v2_lm_gpu.csv",
        "results/summary_e3v2_gpu.md", "results/figures/e3v2_gpu_dynamics.png"]
out = "/kaggle/working/new_results.zip" if os.path.exists("/kaggle/working") else "new_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for k in keep:
        paths = [k] if os.path.isfile(k) else [os.path.join(d, f) for d, _, fs in os.walk(k) for f in fs]
        for p in paths:
            z.write(p)
print(out, f"{os.path.getsize(out) / 2**20:.1f} MiB:", sorted(z.namelist())[:5], "...")